# Leksyon 10 - Mga AI Agent sa Produksyon

Sa leksyong ito matututuhan mo ang **mga pattern ng produksyon** para sa mga AI agent gamit ang Microsoft Agent Framework (Python). Saklaw namin:

- **Observability** — pagdaragdag ng pagtatala ng oras at pag-log sa mga interaksyon ng agent
- **Evaluation** — paggamit ng tagasuri na agent upang markahan ang kalidad ng mga tugon
- **Cost management** — mga estratehiya para sa pag-optimize ng token at pagpili ng modelo

Ang senaryo ay isang **ahente ng paglalakbay** na tumutulong sa mga gumagamit na magplano ng mga biyahe, na may nakapaloob na pagsubaybay at pagtatasa.


## Pagsasaayos


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Mga Pagsasaalang-alang sa Produksyon

Ang paglilipat ng mga AI agent mula sa mga prototype patungo sa produksyon ay nangangailangan ng maingat na pagtutok sa tatlong haligi:

1. **Obserbabilidad** — Kailangan mong magkaroon ng pananaw sa ginagawa ng agent, gaano katagal ito tumatagal, at kung aling mga tool ang tinatawag nito. Kung walang tracing at logging, halos imposible ang pag-debug ng mga isyu sa produksyon.

2. **Ebalwasyon** — Ang mga awtomatikong pagsusuri ng kalidad ay nagsisiguro na ang mga tugon ng agent ay nananatiling tumpak, kumpleto, at kapaki-pakinabang sa paglipas ng panahon. Ang isang evaluator agent ay maaaring magbigay ng iskor sa mga tugon laban sa itinakdang pamantayan.

3. **Pamamahala ng Gastos** — Ang paggamit ng token ay direktang nakakaapekto sa gastos. Ang mga estratehiya tulad ng pag-optimize ng prompt, pagpili ng modelo, at caching ay tumutulong na panatilihing kontrolado ang mga gastusin nang hindi sinasakripisyo ang kalidad.


## Pagbuo ng Isang Nasusubaybay na Ahente

Itinatakda namin ang mga tool para sa paglalakbay at binabalot namin ang tawag sa ahente ng pagtala ng oras upang masubaybayan namin ang pagkaantala. Kapag nasa produksyon, isasama mo ang OpenTelemetry o isang katulad na tracing backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mga Pattern ng Pagsusuri

Isang karaniwang pattern sa produksyon ay ang paggamit ng pangalawang ahente bilang isang **tagasuri**. Binibigyan ng iskor ng tagasuri ang tugon ng pangunahing ahente batay sa mga paunang natukoy na pamantayan tulad ng pagiging kumpleto, katumpakan, at kapaki-pakinabang.

Ito ay nagbibigay-daan sa:
- Mga awtomatikong kontrol sa kalidad bago makarating ang mga tugon sa mga gumagamit
- Pagtuklas ng regresyon kapag nagbago ang mga prompt o mga modelo
- Patuloy na pagmamanman ng pagganap ng ahente sa paglipas ng panahon


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mga Estratehiya sa Pamamahala ng Gastos

Mahalaga ang pagkontrol ng gastos para sa mga AI agent sa produksyon. Narito ang mga pangunahing estratehiya:

| Strategy | Description |
|---|---|
| **Pag-optimize ng prompt** | Panatilihing maikli ang mga tagubilin ng sistema. Alisin ang mga labis na konteksto upang mabawasan ang mga token sa input. |
| **Pagpili ng modelo** | Gumamit ng mas maliliit at mas murang modelo (hal. GPT-4o-mini) para sa mga simpleng gawain tulad ng pag-uuri o pagkuha, at ireserba ang mas malalaking modelo para sa mas kumplikadong pangangatwiran. |
| **Pag-cache** | I-cache ang mga resulta ng tool at mga madalas na query upang maiwasan ang paulit-ulit na mga tawag sa API. |
| **Badyet ng token** | Magtakda ng mga limitasyon ng `max_tokens` upang maiwasan ang hindi inaasahang mahahabang tugon. |
| **Pag-batch** | Pagsama-samahin ang maraming query ng user sa isang tawag sa API kapag posible. |

Sa praktika, mahusay ang isang pinahahating pamamaraan: idirekta ang mga tuwirang kahilingan sa isang mabilis at murang modelo at ipasa lamang ang mga kumplikadong query sa isang mas may kakayahang modelo.


## Buod

Sa leksiyong ito natutunan mo kung paano:

1. **Add observability** sa mga interaksyon ng agent gamit ang pagtatala ng oras at pag-log, na naglalatag ng batayan para sa pagsubaybay at pagmamanman.
2. **Evaluate agent responses** nang awtomatiko gamit ang isang evaluator agent na nagbibigay ng iskor para sa pagiging kumpleto, katumpakan, at pagiging kapaki-pakinabang.
3. **Manage costs** sa pamamagitan ng pag-optimize ng prompt, pagpili ng modelo, caching, at mga badyet ng token.

Tinutulungan ng mga pattern na ito para sa produksyon na matiyak na ang iyong mga AI agent ay maaasahan, nasusukat, at matipid sa malakihang paggamit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:

Ang dokumentong ito ay isinalin gamit ang serbisyong pagsasalin na pinatatakbo ng AI na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kami para sa katumpakan, mangyaring tandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatumpak. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na opisyal na pinagkukunan. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasaling-tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na magmumula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
